# SCX quickstart with PBMC3k
Use your existing Scanpy/AnnData environment. Load PBMC3k, save and read SCX, then view a small TSV table. No SCX package install is needed.

## 1. Load SCX and PBMC3k
Use local PBMC3k data if present. Otherwise, download it once (about 6 MB).
Run from the repo root, `tutorials/`, or your own folder.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import anndata as ad
import numpy as np
import pandas as pd

if not Path("scx.py").exists():
    urlretrieve("https://github.com/junha-shin/scx-format/releases/download/v0.1.0/scx.py", "scx.py")
import scx

candidates = [Path("examples/pbmc3k/data/pbmc3k_raw.h5ad"),
              Path("../examples/pbmc3k/data/pbmc3k_raw.h5ad"), Path("pbmc3k_raw.h5ad")]
data = next((p for p in candidates if p.exists()), candidates[-1])
if not data.exists():
    urlretrieve("https://falexwolf.de/data/pbmc3k_raw.h5ad", data)
adata = ad.read_h5ad(data)
adata.var["symbol"] = adata.var_names
adata.var_names = adata.var["gene_ids"].to_numpy()
print("Input (cells x features):", adata.shape)
display(adata.var.head())


Input (cells x features): (2700, 32738)


,gene_ids,symbol
ENSG00000243485,ENSG00000243485,MIR1302-10
ENSG00000237613,ENSG00000237613,FAM138A
ENSG00000186092,ENSG00000186092,OR4F5
ENSG00000238009,ENSG00000238009,RP11-34P13.7
ENSG00000239945,ENSG00000239945,RP11-34P13.8


## 2. Save and read SCX
This dataset has raw counts in `X`. Select it with `counts="X"` and use `normalized=None`.
Each full run makes a new output folder and keeps earlier results.


In [2]:
import tempfile
run = Path(tempfile.mkdtemp(prefix="scx-pbmc3k-", dir="."))
scx_dir = run / "pbmc3k.scx"
scx.write_scx(adata, scx_dir, counts="X", normalized=None)
restored = scx.read_scx(scx_dir)
print("Restored (cells x features):", restored.shape)
print("Counts layer:", restored.layers["counts"].shape)


Restored (cells x features): (2700, 32738)
Counts layer: (2700, 32738)


## 3. View a small TSV table
Pick five cells and the five genes with the highest total counts.
Only this small table is dense. The full matrix stays sparse.


In [3]:
totals = np.asarray(adata.X.sum(axis=0)).ravel()
features = adata.var_names[np.argsort(-totals)[:5]].tolist()
preview_path = run / "preview.tsv"
scx.to_tsv(scx_dir, preview_path, cells=adata.obs_names[:5].tolist(), features=features)
preview = pd.read_csv(preview_path, sep="	", index_col="cell_id")
print("TSV preview (cells x features):", preview.shape)
display(preview.head())


TSV preview (cells x features): (5, 5)


,ENSG00000251562,ENSG00000205542,ENSG00000166710,ENSG00000147403,ENSG00000167526
cell_id,,,,,
AAACATACAACCAC-1,49,47,76,34,29
AAACATTGAGCTAC-1,142,62,75,92,45
AAACATTGATCAGC-1,171,117,69,49,16
AAACCGTGCTTCCG-1,11,114,41,22,15
AAACCGTGTATGCG-1,22,21,35,8,4


Expected sizes: input and restored data **2700 x 32738**, TSV **5 x 5**.
SCX and TSV use cells x features.

For your own data, replace step 1 with your `adata` object.
The default reads counts from `layers['counts']` and normalized values from `X`.
Keep your original object.

To repeat a write step, make a new output folder in step 2 first.
Existing output files are protected.

[R guide](quickstart.md) | [Detailed options](detailed.ipynb) | [Common errors](../docs/first-use.md)

Data: [10x PBMC3k, shared by Scanpy](../examples/pbmc3k/README.md). The dataset has its own terms of use.
